# 03 - Univariate Feature Selection

This notebook evaluates probabilistic classifiers after applying a univariate
filter feature-selection strategy.

The objective is to assess whether reducing the original predictor set can
improve predictive performance while retaining only variables with strong
individual relationships with the target.

Because the dataset contains different predictor types, the univariate score is
adapted to the type of each variable:

- Continuous predictors are evaluated using ANOVA F-statistics.
- Ordinal and nominal categorical predictors are evaluated using Mutual
  Information.

The final subset contains the 15 highest-ranked original predictors.

Feature selection is performed inside the cross-validation procedure to avoid
information leakage.

In [1]:
import pandas as pd
import numpy as np

from scipy.special import logsumexp

from sklearn.base import (
    BaseEstimator,
    TransformerMixin,
    ClassifierMixin,
    clone
)

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    f_classif,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Dataset Preparation

The same dataset preparation and target encoding used in the probabilistic
baseline are retained.

No information from the validation folds is used during feature selection.

In [2]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Predictor Types

Predictor types are defined in the same way as in the previous notebooks.

Continuous predictors are scored using ANOVA, while ordinal and nominal
categorical predictors are scored using Mutual Information.

In [3]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

discrete_features = (
    ordinal_features
    + categorical_features
)

print("Continuous:", len(continuous_features))
print("Discrete/ordinal:", len(discrete_features))
print("Total:", len(continuous_features) + len(discrete_features))

Continuous: 14
Discrete/ordinal: 16
Total: 30


## 3. Type-Aware Univariate Selector

The selector ranks original predictors independently.

For continuous variables, the ANOVA F-statistic is used.

For ordinal and categorical predictors, Mutual Information with the target is
used.

Scores are standardized separately within each scoring family before being
combined, because ANOVA and Mutual Information are measured on different
scales.

The 15 original variables with the highest standardized scores are retained.

In [6]:
class TypeAwareUnivariateSelector(
    BaseEstimator,
    TransformerMixin
):
    def __init__(
        self,
        continuous_features,
        discrete_features,
        k=15,
        random_state=42
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.k = k
        self.random_state = random_state

    def fit(self, X, y):

        X = X.copy()

        score_rows = []

        # ----------------------------
        # Continuous variables: ANOVA
        # ----------------------------
        if len(self.continuous_features) > 0:

            X_cont = X[
                self.continuous_features
            ].astype(float)

            anova_scores, _ = f_classif(
                X_cont,
                y
            )

            for feature, score in zip(
                self.continuous_features,
                anova_scores
            ):
                score_rows.append({
                    "Feature": feature,
                    "Method": "ANOVA",
                    "RawScore": score
                })

        # -----------------------------------
        # Discrete variables: Mutual Info
        # -----------------------------------
        if len(self.discrete_features) > 0:

            X_disc = X[
                self.discrete_features
            ].copy()

            for feature in self.discrete_features:

                # Convert nominal/string variables to integer codes.
                # Ordinal predictors are already numeric and are kept
                # in their original integer representation.
                if not pd.api.types.is_numeric_dtype(
                    X_disc[feature]
                ):

                    X_disc[feature] = pd.factorize(
                        X_disc[feature]
                    )[0]

            X_disc = X_disc.astype(float)

            mi_scores = mutual_info_classif(
                X_disc,
                y,
                discrete_features=True,
                random_state=self.random_state
            )

            for feature, score in zip(
                self.discrete_features,
                mi_scores
            ):
                score_rows.append({
                    "Feature": feature,
                    "Method": "Mutual Information",
                    "RawScore": score
                })

        scores = pd.DataFrame(
            score_rows
        )

        # Standardize separately because
        # ANOVA and MI use different scales.
        scores["StandardizedScore"] = 0.0

        for method in scores["Method"].unique():

            mask = (
                scores["Method"] == method
            )

            values = scores.loc[
                mask,
                "RawScore"
            ]

            std = values.std(ddof=0)

            if std == 0:

                standardized = np.zeros(
                    len(values)
                )

            else:

                standardized = (
                    values - values.mean()
                ) / std

            scores.loc[
                mask,
                "StandardizedScore"
            ] = standardized

        scores = scores.sort_values(
            "StandardizedScore",
            ascending=False
        )

        self.scores_ = scores.reset_index(
            drop=True
        )

        self.selected_features_ = (
            self.scores_
            .head(self.k)["Feature"]
            .tolist()
        )

        return self

    def transform(self, X):

        return X[
            self.selected_features_
        ].copy()

    def get_feature_names_out(
        self,
        input_features=None
    ):

        return np.array(
            self.selected_features_
        )

## 4. Descriptive Full-Data Ranking

A selector is fitted once on the complete dataset only to inspect the resulting
ranking of variables.

This ranking is descriptive and is not used for performance estimation.

During cross-validation, feature selection will be fitted independently inside
each training fold.

In [7]:
descriptive_selector = TypeAwareUnivariateSelector(
    continuous_features=continuous_features,
    discrete_features=discrete_features,
    k=15,
    random_state=42
)

descriptive_selector.fit(
    X,
    y
)

display(
    descriptive_selector.scores_
    .round(4)
)

print("\nSelected variables:")
print(
    descriptive_selector.selected_features_
)

,Feature,Method,RawScore,StandardizedScore
0,JobRole,Mutual Information,0.0302,2.0206
1,OverTime,Mutual Information,0.0277,1.7615
2,TotalWorkingYears,ANOVA,44.2525,1.5255
3,JobLevel,Mutual Information,0.0250,1.4868
4,YearsInCurrentRole,ANOVA,38.8383,1.2148
5,MonthlyIncome,ANOVA,38.4888,1.1947
6,Age,ANOVA,38.1759,1.1767
7,YearsWithCurrManager,ANOVA,36.7123,1.0927
8,StockOptionLevel,Mutual Information,0.0210,1.0813
9,YearsAtCompany,ANOVA,27.0016,0.5354



Selected variables:
['JobRole', 'OverTime', 'TotalWorkingYears', 'JobLevel', 'YearsInCurrentRole', 'MonthlyIncome', 'Age', 'YearsWithCurrManager', 'StockOptionLevel', 'YearsAtCompany', 'MaritalStatus', 'JobInvolvement', 'BusinessTravel', 'EnvironmentSatisfaction', 'JobSatisfaction']


## 5. Mixed Naive Bayes

The same Mixed Naive Bayes formulation used in the baseline experiment is
retained.

Only the variables selected within each training fold are provided to the
classifier.

In [8]:
class MixedNaiveBayes(
    BaseEstimator,
    ClassifierMixin
):
    def __init__(
        self,
        continuous_features,
        discrete_features,
        alpha=1.0,
        var_smoothing=1e-9
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.alpha = alpha
        self.var_smoothing = var_smoothing

    def fit(self, X, y):

        X = X.copy()
        y = np.asarray(y)

        self.classes_ = np.unique(y)

        selected_columns = X.columns.tolist()

        self.active_continuous_features_ = [
            feature
            for feature in self.continuous_features
            if feature in selected_columns
        ]

        self.active_discrete_features_ = [
            feature
            for feature in self.discrete_features
            if feature in selected_columns
        ]

        self.class_prior_ = {}
        self.gaussian_parameters_ = {}
        self.discrete_probabilities_ = {}
        self.discrete_default_probabilities_ = {}

        n_samples = len(y)

        for cls in self.classes_:

            class_mask = (y == cls)
            X_cls = X.loc[class_mask]

            class_count = class_mask.sum()

            self.class_prior_[cls] = (
                class_count / n_samples
            )

            self.gaussian_parameters_[cls] = {}

            for feature in self.active_continuous_features_:

                values = (
                    X_cls[feature]
                    .astype(float)
                    .to_numpy()
                )

                mean = values.mean()
                variance = (
                    values.var()
                    + self.var_smoothing
                )

                self.gaussian_parameters_[
                    cls
                ][feature] = (
                    mean,
                    variance
                )

            self.discrete_probabilities_[cls] = {}
            self.discrete_default_probabilities_[cls] = {}

            for feature in self.active_discrete_features_:

                all_categories = (
                    X[feature]
                    .astype(str)
                    .unique()
                )

                class_values = (
                    X_cls[feature]
                    .astype(str)
                )

                counts = (
                    class_values
                    .value_counts()
                )

                n_categories = (
                    len(all_categories) + 1
                )

                denominator = (
                    class_count
                    + self.alpha * n_categories
                )

                probabilities = {}

                for category in all_categories:

                    probabilities[category] = (
                        counts.get(
                            category,
                            0
                        )
                        + self.alpha
                    ) / denominator

                self.discrete_probabilities_[
                    cls
                ][feature] = probabilities

                self.discrete_default_probabilities_[
                    cls
                ][feature] = (
                    self.alpha / denominator
                )

        return self

    def _joint_log_likelihood(self, X):

        X = X.copy()

        joint_log_likelihood = []

        for cls in self.classes_:

            log_probability = np.full(
                X.shape[0],
                np.log(
                    self.class_prior_[cls]
                ),
                dtype=float
            )

            for feature in self.active_continuous_features_:

                mean, variance = (
                    self.gaussian_parameters_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(float)
                    .to_numpy()
                )

                log_probability += (
                    -0.5 * np.log(
                        2 * np.pi * variance
                    )
                    - (
                        (values - mean) ** 2
                        / (2 * variance)
                    )
                )

            for feature in self.active_discrete_features_:

                probabilities = (
                    self.discrete_probabilities_[
                        cls
                    ][feature]
                )

                default_probability = (
                    self.discrete_default_probabilities_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(str)
                    .to_numpy()
                )

                feature_probabilities = np.array([
                    probabilities.get(
                        value,
                        default_probability
                    )
                    for value in values
                ])

                log_probability += np.log(
                    feature_probabilities
                )

            joint_log_likelihood.append(
                log_probability
            )

        return np.column_stack(
            joint_log_likelihood
        )

    def predict_proba(self, X):

        joint_log_likelihood = (
            self._joint_log_likelihood(X)
        )

        log_normalizer = logsumexp(
            joint_log_likelihood,
            axis=1,
            keepdims=True
        )

        return np.exp(
            joint_log_likelihood
            - log_normalizer
        )

    def predict(self, X):

        probabilities = self.predict_proba(X)

        return self.classes_[
            np.argmax(
                probabilities,
                axis=1
            )
        ]

## 6. Cross-Validation Evaluation

For every cross-validation fold:

1. The univariate selector is fitted using only the training data.
2. The 15 highest-ranked original predictors are selected.
3. Model-specific preprocessing is created using only those selected variables.
4. Each classifier is fitted on the training partition.
5. Predictions and posterior probabilities are obtained for the validation
   partition.
6. The evaluation metrics are calculated.

This procedure prevents feature-selection leakage.

In [9]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [10]:
def multiclass_brier_score(
    y_true,
    probabilities,
    classes
):

    y_true = np.asarray(
        y_true
    )

    indicator_matrix = np.zeros_like(
        probabilities,
        dtype=float
    )

    for index, cls in enumerate(classes):

        indicator_matrix[:, index] = (
            y_true == cls
        ).astype(float)

    squared_error = (
        probabilities
        - indicator_matrix
    ) ** 2

    return np.mean(
        squared_error.sum(axis=1)
    )

## 7. Model-Specific Preprocessing

After feature selection, preprocessing is dynamically adapted to the selected
variables.

Logistic Regression uses one-hot encoding for selected nominal predictors.

LDA and QDA use one numerical column per selected original predictor to avoid
the covariance-rank problems observed with expanded dummy representations.

In [11]:
def create_preprocessors(
    selected_features
):

    selected_continuous = [
        feature
        for feature in continuous_features
        if feature in selected_features
    ]

    selected_ordinal = [
        feature
        for feature in ordinal_features
        if feature in selected_features
    ]

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in selected_features
    ]

    selected_numeric = (
        selected_continuous
        + selected_ordinal
    )

    logistic_preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    selected_categorical
                )
            ]
        )
    )

    discriminant_preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    ),
                    selected_categorical
                )
            ]
        )
    )

    return (
        logistic_preprocessor,
        discriminant_preprocessor,
        selected_continuous,
        selected_ordinal,
        selected_categorical
    )

In [12]:
def evaluate_fold_model(
    name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    fold
):

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    probabilities = (
        model.predict_proba(
            X_test
        )
    )

    classes = model.classes_

    positive_index = np.where(
        classes == 1
    )[0][0]

    positive_probability = (
        probabilities[
            :,
            positive_index
        ]
    )

    return {
        "Model": name,
        "Fold": fold,

        "Accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            y_pred,
            pos_label=1
        ),

        "Specificity": recall_score(
            y_test,
            y_pred,
            pos_label=0
        ),

        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "AUC": roc_auc_score(
            y_test,
            positive_probability
        ),

        "Brier": multiclass_brier_score(
            y_test,
            probabilities,
            classes
        )
    }

## 8. Univariate Feature-Selection Experiment

The complete univariate feature-selection and model-evaluation process is now
repeated independently for each of the ten stratified folds.

The selected variable names are also stored to analyse selection stability.

In [13]:
fold_results = []
selection_records = []

for fold, (train_index, test_index) in enumerate(
    cv.split(X, y),
    start=1
):

    print(f"Fold {fold}/10")

    X_train = (
        X.iloc[train_index]
        .copy()
    )

    X_test = (
        X.iloc[test_index]
        .copy()
    )

    y_train = (
        y.iloc[train_index]
        .copy()
    )

    y_test = (
        y.iloc[test_index]
        .copy()
    )

    selector = TypeAwareUnivariateSelector(
        continuous_features=continuous_features,
        discrete_features=discrete_features,
        k=15,
        random_state=42
    )

    selector.fit(
        X_train,
        y_train
    )

    selected_features = (
        selector.selected_features_
    )

    selection_records.append({
        "Fold": fold,
        "SelectedFeatures": selected_features
    })

    X_train_selected = (
        selector.transform(
            X_train
        )
    )

    X_test_selected = (
        selector.transform(
            X_test
        )
    )

    (
        logistic_preprocessor,
        discriminant_preprocessor,
        selected_continuous,
        selected_ordinal,
        selected_categorical
    ) = create_preprocessors(
        selected_features
    )

    selected_discrete = (
        selected_ordinal
        + selected_categorical
    )

    # ------------------------
    # Mixed Naive Bayes
    # ------------------------

    mixed_nb = MixedNaiveBayes(
        continuous_features=selected_continuous,
        discrete_features=selected_discrete,
        alpha=1.0
    )

    fold_results.append(
        evaluate_fold_model(
            "Mixed Naive Bayes",
            mixed_nb,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # ------------------------
    # Logistic Regression
    # ------------------------

    logistic_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    logistic_preprocessor
                )
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "Logistic Regression",
            logistic_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # ------------------------
    # LDA
    # ------------------------

    lda_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    discriminant_preprocessor
                )
            ),
            (
                "classifier",
                LinearDiscriminantAnalysis()
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "LDA",
            lda_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

    # ------------------------
    # QDA
    # ------------------------

    qda_model = Pipeline(
        steps=[
            (
                "preprocessing",
                clone(
                    discriminant_preprocessor
                )
            ),
            (
                "classifier",
                QuadraticDiscriminantAnalysis()
            )
        ]
    )

    fold_results.append(
        evaluate_fold_model(
            "QDA",
            qda_model,
            X_train_selected,
            X_test_selected,
            y_train,
            y_test,
            fold
        )
    )

print("\nEvaluation completed.")

Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

Evaluation completed.


In [14]:
univariate_fold_results = pd.DataFrame(
    fold_results
)

display(
    univariate_fold_results.round(4)
)

,Model,Fold,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,1,0.7347,0.3171,0.5417,0.7724,0.4000,0.7883,0.3449
1,Logistic Regression,1,0.8707,0.7778,0.2917,0.9837,0.4242,0.7304,0.2327
2,LDA,1,0.8639,0.7500,0.2500,0.9837,0.3750,0.7368,0.2351
3,QDA,1,0.8299,0.4615,0.2500,0.9431,0.3243,0.7066,0.2579
4,Mixed Naive Bayes,2,0.7143,0.2955,0.5417,0.7480,0.3824,0.7033,0.3733
5,Logistic Regression,2,0.8095,0.3571,0.2083,0.9268,0.2632,0.8096,0.2231
6,LDA,2,0.8163,0.3846,0.2083,0.9350,0.2703,0.8117,0.2295
7,QDA,2,0.8503,0.5417,0.5417,0.9106,0.5417,0.8072,0.2456
8,Mixed Naive Bayes,3,0.7347,0.3171,0.5417,0.7724,0.4000,0.6697,0.3681
9,Logistic Regression,3,0.8435,0.5714,0.1667,0.9756,0.2581,0.7852,0.2165


## 9. Cross-Validation Summary

The mean and standard deviation of each evaluation metric are calculated across
the ten folds for every classifier.

In [15]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

summary_rows = []

for model_name in univariate_fold_results["Model"].unique():

    model_data = (
        univariate_fold_results[
            univariate_fold_results["Model"]
            == model_name
        ]
    )

    row = {
        "Model": model_name
    }

    for metric in metrics:

        row[f"{metric}_mean"] = (
            model_data[metric]
            .mean()
        )

        row[f"{metric}_std"] = (
            model_data[metric]
            .std()
        )

    summary_rows.append(
        row
    )

univariate_results = pd.DataFrame(
    summary_rows
)

display(
    univariate_results.round(4)
)

,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,F1_mean,F1_std,AUC_mean,AUC_std,Brier_mean,Brier_std
0,Mixed Naive Bayes,0.7707,0.0357,0.3733,0.0578,0.6033,0.0873,0.8029,0.0354,0.4600,0.0648,0.7633,0.0584,0.3190,0.0418
1,Logistic Regression,0.8578,0.0273,0.6676,0.1937,0.2708,0.1087,0.9708,0.0251,0.3736,0.1329,0.7919,0.0468,0.2112,0.0240
2,LDA,0.8565,0.0280,0.6328,0.1896,0.2658,0.1070,0.9700,0.0179,0.3700,0.1307,0.7714,0.0404,0.2239,0.0238
3,QDA,0.8320,0.0147,0.4728,0.0539,0.3924,0.1048,0.9165,0.0187,0.4240,0.0778,0.7302,0.0527,0.2628,0.0243


In [16]:
univariate_summary = univariate_results[
    [
        "Model",
        "Accuracy_mean",
        "Precision_mean",
        "Recall_mean",
        "Specificity_mean",
        "F1_mean",
        "AUC_mean",
        "Brier_mean"
    ]
].copy()

univariate_summary.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

univariate_summary = (
    univariate_summary
    .sort_values(
        "F1",
        ascending=False
    )
)

display(
    univariate_summary.round(4)
)

,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,0.7707,0.3733,0.6033,0.8029,0.4600,0.7633,0.3190
3,QDA,0.8320,0.4728,0.3924,0.9165,0.4240,0.7302,0.2628
1,Logistic Regression,0.8578,0.6676,0.2708,0.9708,0.3736,0.7919,0.2112
2,LDA,0.8565,0.6328,0.2658,0.9700,0.3700,0.7714,0.2239


## 10. Selection Stability

Because feature selection is fitted independently in every training fold, the
selected subset can vary slightly across folds.

The frequency with which each original predictor is selected is therefore
examined.

In [17]:
selection_frequency = {
    feature: 0
    for feature in X.columns
}

for record in selection_records:

    for feature in record[
        "SelectedFeatures"
    ]:

        selection_frequency[
            feature
        ] += 1

selection_frequency_df = (
    pd.DataFrame({
        "Feature":
            selection_frequency.keys(),

        "SelectedFolds":
            selection_frequency.values()
    })
    .sort_values(
        [
            "SelectedFolds",
            "Feature"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    selection_frequency_df
)

,Feature,SelectedFolds
0,Age,10
1,BusinessTravel,10
2,JobInvolvement,10
3,JobLevel,10
4,JobRole,10
5,MaritalStatus,10
6,MonthlyIncome,10
7,OverTime,10
8,StockOptionLevel,10
9,TotalWorkingYears,10


## 11. Comparison with the Full-Feature Baseline

The univariate experiment is compared with the probabilistic baseline obtained
using all 30 valid original predictors.

This comparison shows whether reducing the predictor set improves or degrades
each classifier.

In [18]:
baseline_summary = pd.read_csv(
    "../../results/pa2/baseline_summary.csv"
)

comparison = (
    baseline_summary[
        [
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .merge(
        univariate_summary[
            [
                "Model",
                "F1",
                "Recall",
                "AUC",
                "Brier"
            ]
        ],
        on="Model",
        suffixes=(
            "_Baseline",
            "_Univariate"
        )
    )
)

comparison["F1_change"] = (
    comparison["F1_Univariate"]
    - comparison["F1_Baseline"]
)

comparison["Recall_change"] = (
    comparison["Recall_Univariate"]
    - comparison["Recall_Baseline"]
)

comparison["AUC_change"] = (
    comparison["AUC_Univariate"]
    - comparison["AUC_Baseline"]
)

comparison["Brier_change"] = (
    comparison["Brier_Univariate"]
    - comparison["Brier_Baseline"]
)

display(
    comparison.round(4)
)

,Model,F1_Baseline,Recall_Baseline,AUC_Baseline,Brier_Baseline,F1_Univariate,Recall_Univariate,AUC_Univariate,Brier_Univariate,F1_change,Recall_change,AUC_change,Brier_change
0,Logistic Regression,0.5373,0.4259,0.8375,0.1857,0.3736,0.2708,0.7919,0.2112,-0.1638,-0.1551,-0.0455,0.0255
1,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902,0.4600,0.6033,0.7633,0.3190,-0.0483,-0.0420,-0.0218,0.0288
2,LDA,0.4256,0.3168,0.8146,0.2036,0.3700,0.2658,0.7714,0.2239,-0.0557,-0.0511,-0.0432,0.0204
3,QDA,0.3946,0.3755,0.7211,0.2786,0.4240,0.3924,0.7302,0.2628,0.0294,0.0168,0.0091,-0.0157


## 12. Export Results

The fold-level performance, aggregated results, and selection frequencies are
stored for the final PA2 comparison.

In [19]:
univariate_fold_results.to_csv(
    "../../results/pa2/univariate_fold_results.csv",
    index=False
)

univariate_results.to_csv(
    "../../results/pa2/univariate_results.csv",
    index=False
)

univariate_summary.to_csv(
    "../../results/pa2/univariate_summary.csv",
    index=False
)

selection_frequency_df.to_csv(
    "../../results/pa2/univariate_selection_frequency.csv",
    index=False
)

print(
    "PA2 univariate results saved successfully."
)

PA2 univariate results saved successfully.


## 13. Main Observations

The univariate filter reduces the predictor set from 30 original variables to
15 variables within each training fold.

Mixed Naive Bayes obtains the highest F1-measure under the univariate strategy
(0.4600) and also the highest Recall (0.6033). However, both values are lower
than its full-feature baseline, and its AUC decreases from 0.7851 to 0.7633
while its Brier score worsens from 0.2902 to 0.3190.

QDA is the only classifier that improves relative to its baseline across all
main comparison metrics. Its F1 increases from 0.3946 to 0.4240, Recall from
0.3755 to 0.3924, and AUC from 0.7211 to 0.7302, while its Brier score improves
from 0.2786 to 0.2628.

Logistic Regression is the classifier most negatively affected by the
univariate reduction. Its F1 decreases from 0.5373 to 0.3736 and its Recall
from 0.4259 to 0.2708. Its AUC also decreases and its Brier score becomes
worse.

LDA follows a similar pattern, with lower F1, Recall and AUC and a slightly
higher Brier score after univariate selection.

The selection process is relatively stable. Thirteen variables are selected
in all ten folds, including `Age`, `BusinessTravel`, `JobInvolvement`,
`JobLevel`, `JobRole`, `MaritalStatus`, `MonthlyIncome`, `OverTime`,
`StockOptionLevel`, `TotalWorkingYears`, `YearsAtCompany`,
`YearsInCurrentRole`, and `YearsWithCurrManager`.

Overall, the univariate filter does not improve the strongest probabilistic
baseline configuration, although QDA benefits from the reduced representation.